In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.special import logsumexp


In [ ]:
def bayesian_posterior_update(nu_0, Gamma_0, X_cumul, y_cumul, sigma2):
    """
    Compute the Bayesian posterior for a normal linear regression model,
    given the *initial* prior and *cumulative* data through week w-2.

    Model:   y = X^T theta + epsilon,   epsilon ~ N(0, sigma^2)
    Prior:   theta ~ N(nu_0, Gamma_0)

    Posterior at week w-2 (w >= 3):

        Gamma_{w-2} = ( Gamma_0^{-1}  +  X_{w-2}^T X_{w-2} / sigma^2 )^{-1}
        nu_{w-2}    = Gamma_{w-2} ( Gamma_0^{-1} nu_0
                                    +  X_{w-2}^T y_{w-2} / sigma^2 )

    where X_{w-2} is the (n, p) design matrix that vertically stacks all
    observation rows from weeks 1 through w-2, and y_{w-2} is the
    corresponding (n,) response vector.

    At w = 2 the posterior equals the prior (no data yet), so the caller
    should simply pass (nu_0, Gamma_0) directly to estimate_belief_state.

    Parameters
    ----------
    nu_0     : (p,) array   – initial prior mean  nu_0
    Gamma_0  : (p, p) array – initial prior covariance  Gamma_0
    X_cumul  : (n, p) array – cumulative design matrix  X_{w-2}
                              (rows from weeks 1 .. w-2)
    y_cumul  : (n,) array   – cumulative response vector  y_{w-2}
                              (responses from weeks 1 .. w-2)
    sigma2   : float        – observation noise variance  (sigma)^2

    Returns
    -------
    nu_post    : (p,) array   – posterior mean       nu_{w-2}
    Gamma_post : (p, p) array – posterior covariance  Gamma_{w-2}
    """
    X = np.atleast_2d(X_cumul)           # (n, p)
    y = np.atleast_1d(y_cumul).ravel()   # (n,)

    Gamma_0_inv = np.linalg.inv(Gamma_0)
    Gamma_post_inv = Gamma_0_inv + (1.0 / sigma2) * (X.T @ X)
    Gamma_post = np.linalg.inv(Gamma_post_inv)
    nu_post = Gamma_post @ (Gamma_0_inv @ nu_0 + (1.0 / sigma2) * (X.T @ y))

    return nu_post, Gamma_post


def estimate_belief_state(
    w, J, y_hat_prev, v_hat_prev,
    nu_MY, Gamma_MY, sigma2_MY,
    nu_Y, Gamma_Y, sigma2_Y,
    nu_tilde_Y, Gamma_tilde_Y, sigma2_tilde_Y,
    X_MY, M_Y_obs,
    X_Y, X_tilde_Y,
    I_w, J_w,
    Y_w=None, tilde_Y_w=None,
    rng=None,
):
    """
    Estimate belief state b_w(y) on Sunday night of week w-1  (w >= 2).

    Implements the particle filter (sequential Monte Carlo) from Algorithm 1.

    For each particle j = 1..J:
      1. Draw parameter particles theta^{MY_{dt},(j)}, theta^{Y,(j)},
         theta^{tilde_Y,(j)} from their posteriors at week w-2.
      2. Draw y_w^{(j)} ~ N(X_Y^T theta^{Y,(j)}, sigma_Y^2).
      3. If Y_w is observed (I_w=1, J_w=1):
           set y_w^{(j)} = Y_w and weight by mediator + Y likelihoods.
         Elif only tilde_Y_w is observed (I_w=0, J_w=1):
           weight by mediator + tilde_Y likelihoods.
         Else:
           weight by mediator likelihoods only.
    Then normalise weights, compute ESS, and resample if ESS < 0.5*J.

    Parameters
    ----------
    w            : int  – current week (>= 2)
    J            : int  – number of particles
    y_hat_prev   : (J, w-1) array – particle trajectories hat{y}_{1:w-1}
    v_hat_prev   : (J,) array     – normalised weights hat{v}_{w-1}

    nu_MY        : list[ndarray]  – posterior means  nu_{w-2}^{MY_{dt}}, one per mediator
    Gamma_MY     : list[ndarray]  – posterior covs   Gamma_{w-2}^{MY_{dt}}
    sigma2_MY    : list[float]    – noise variances  (sigma^{MY_{dt}})^2
                   All three lists have length n_med (= 12 for d=1..6, t=1,2).

    nu_Y         : (p_Y,) array   – posterior mean   nu_{w-2}^Y
    Gamma_Y      : (p_Y, p_Y)     – posterior cov    Gamma_{w-2}^Y
    sigma2_Y     : float           – noise variance  (sigma^Y)^2

    nu_tilde_Y   : (p_tY,) array   – posterior mean  nu_{w-2}^{tilde_Y}
    Gamma_tilde_Y: (p_tY, p_tY)    – posterior cov   Gamma_{w-2}^{tilde_Y}
    sigma2_tilde_Y: float           – noise variance (sigma^{tilde_Y})^2

    X_MY         : list[ndarray]   – design vectors  X_{w-1}^{MY_{dt}}, length n_med
    M_Y_obs      : list[float]     – observed mediators M_{w-1,d,t}^Y (NaN = missing)
    X_Y          : (p_Y,) array    – design vector   X_{w-1}^Y
    X_tilde_Y    : (p_tY,) array   – design vector   X_{w-1}^{tilde_Y}

    I_w          : int   – 1 if Y_w is fully observed, 0 otherwise
    J_w          : int   – 1 if tilde_Y_w (or Y_w) is at least partially observed
    Y_w          : float or None – observed primary outcome  (required when I_w=1)
    tilde_Y_w    : float or None – observed proxy outcome    (required when I_w=0, J_w=1)

    rng          : numpy.random.Generator or None

    Returns
    -------
    y_hat_new    : (J, w) array – updated trajectories hat{y}_{1:w}
    v_hat_new    : (J,) array   – updated normalised weights hat{v}_w
    """
    if rng is None:
        rng = np.random.default_rng()

    n_med = len(nu_MY)

    # ── Step 1: draw parameter particles (vectorised over j = 1..J) ──

    # theta^{MY_{dt},(j)}, each entry shape (J, p_m)
    theta_MY = [
        rng.multivariate_normal(nu_MY[m], Gamma_MY[m], size=J)
        for m in range(n_med)
    ]
    # theta^{Y,(j)}, shape (J, p_Y)
    theta_Y = rng.multivariate_normal(nu_Y, Gamma_Y, size=J)
    # theta^{tilde_Y,(j)}, shape (J, p_tY)
    theta_tY = rng.multivariate_normal(nu_tilde_Y, Gamma_tilde_Y, size=J)

    # y_w^{(j)} ~ N( X_Y^T theta^{Y,(j)},  (sigma^Y)^2 )
    mu_y = theta_Y @ X_Y                                    # (J,)
    y_w = rng.normal(mu_y, np.sqrt(sigma2_Y))                # (J,)

    # ── Step 2: mediator log-likelihood (all particles at once) ──

    log_med_lik = np.zeros(J)
    for m in range(n_med):
        if np.isnan(M_Y_obs[m]):
            continue
        mu_m = theta_MY[m] @ X_MY[m]                        # (J,)
        log_med_lik += norm.logpdf(
            M_Y_obs[m], loc=mu_m, scale=np.sqrt(sigma2_MY[m])
        )

    # ── Step 3: weight update (observation-regime dependent) ──

    log_w_prev = np.log(np.maximum(v_hat_prev, 1e-300))

    if I_w == 1 and J_w == 1:
        # Y_w is fully observed -> override drawn y_w
        y_w[:] = Y_w
        mu_Y_pred = theta_Y @ X_Y                           # (J,)
        log_Y_lik = norm.logpdf(
            Y_w, loc=mu_Y_pred, scale=np.sqrt(sigma2_Y)
        )                                                    # (J,)
        log_v_tilde = log_w_prev + log_med_lik + log_Y_lik

    elif I_w == 0 and J_w == 1:
        # only proxy tilde_Y_w is observed
        mu_tY = theta_tY @ X_tilde_Y                        # (J,)
        log_tY_lik = norm.logpdf(
            tilde_Y_w, loc=mu_tY, scale=np.sqrt(sigma2_tilde_Y)
        )
        log_v_tilde = log_w_prev + log_med_lik + log_tY_lik

    else:
        # neither Y_w nor tilde_Y_w observed
        log_v_tilde = log_w_prev + log_med_lik

    # ── Step 4: normalise ──

    v_norm = np.exp(log_v_tilde - logsumexp(log_v_tilde))    # (J,)

    # ── Step 5: effective sample size ──

    ESS = 1.0 / np.sum(v_norm ** 2)

    # ── Step 6: resample if ESS < 0.5 * J ──

    y_hat_new = np.zeros((J, w))

    if ESS < 0.5 * J:
        idx = rng.choice(J, size=J, replace=True, p=v_norm)
        y_hat_new[:, :w - 1] = y_hat_prev[idx]
        y_hat_new[:, w - 1] = y_w[idx]
        v_hat_new = np.full(J, 1.0 / J)
    else:
        y_hat_new[:, :w - 1] = y_hat_prev
        y_hat_new[:, w - 1] = y_w
        v_hat_new = v_norm.copy()

    return y_hat_new, v_hat_new


In [ ]:
def estimate_belief_state_minus(
    w, J,
    y_hat_minus_prev, y_hat_plus_prev, v_hat_plus_prev,
    nu_MY, Gamma_MY, sigma2_MY,
    nu_Y, Gamma_Y, sigma2_Y,
    X_MY, M_Y_obs, X_Y,
    rng=None,
):
    """
    Estimate belief state b_w^-(y) on Sunday *evening* of week w-1  (w >= 2).

    For each particle j = 1..J:
      1. Draw theta^{MY_{dt},(j)} from posterior at week w-2.
      2. Draw theta^{Y,(j)} from posterior at week w-2.
      3. Draw y_w^{-,(j)} ~ N(X_Y^T theta^{Y,(j)}, sigma_Y^2).
      4. Weight by mediator likelihoods only:
           v_tilde = v_hat^+_{w-1} * prod phi(M_{w-1,d,t}; ...).
    Then normalise, compute ESS, and resample if ESS < 0.5*J.
    Resampling affects both minus and plus trajectories jointly.

    Parameters
    ----------
    w                 : int  – current week (>= 2)
    J                 : int  – number of particles
    y_hat_minus_prev  : (J, w-1) array – minus trajectories  hat{y}^-_{1:w-1}
    y_hat_plus_prev   : (J, w-1) array – plus  trajectories  hat{y}^+_{1:w-1}
    v_hat_plus_prev   : (J,) array     – normalised weights  hat{v}^+_{w-1}

    nu_MY       : list[ndarray]  – posterior means   nu_{w-2}^{MY_{dt}}
    Gamma_MY    : list[ndarray]  – posterior covs    Gamma_{w-2}^{MY_{dt}}
    sigma2_MY   : list[float]    – noise variances   (sigma^{MY_{dt}})^2

    nu_Y        : (p_Y,) array   – posterior mean    nu_{w-2}^Y
    Gamma_Y     : (p_Y, p_Y)     – posterior cov     Gamma_{w-2}^Y
    sigma2_Y    : float           – noise variance   (sigma^Y)^2

    X_MY        : list[ndarray]  – design vectors    X_{w-1}^{MY_{dt}}
    M_Y_obs     : list[float]    – observed mediators M_{w-1,d,t}^Y  (NaN = missing)
    X_Y         : (p_Y,) array   – design vector     X_{w-1}^Y

    rng         : numpy.random.Generator or None

    Returns
    -------
    y_hat_minus_new : (J, w)   array – minus trajectories extended to week w
    y_hat_plus_out  : (J, w-1) array – plus trajectories through w-1 (possibly resampled)
    v_hat_minus_new : (J,)     array – normalised weights  hat{v}^-_w
    """
    if rng is None:
        rng = np.random.default_rng()

    n_med = len(nu_MY)

    # ── Step 1: draw parameter particles ──

    theta_MY = [
        rng.multivariate_normal(nu_MY[m], Gamma_MY[m], size=J)
        for m in range(n_med)
    ]
    theta_Y = rng.multivariate_normal(nu_Y, Gamma_Y, size=J)      # (J, p_Y)

    # ── Step 2: draw y_w^{-,(j)} ──

    mu_y = theta_Y @ X_Y                                           # (J,)
    y_w_minus = rng.normal(mu_y, np.sqrt(sigma2_Y))                 # (J,)

    # ── Step 3: mediator log-likelihood ──

    log_med_lik = np.zeros(J)
    for m in range(n_med):
        if np.isnan(M_Y_obs[m]):
            continue
        mu_m = theta_MY[m] @ X_MY[m]                               # (J,)
        log_med_lik += norm.logpdf(
            M_Y_obs[m], loc=mu_m, scale=np.sqrt(sigma2_MY[m])
        )

    # ── Step 4: weight update (mediators only) ──

    log_w_prev = np.log(np.maximum(v_hat_plus_prev, 1e-300))
    log_v_tilde = log_w_prev + log_med_lik

    # ── Step 5: normalise ──

    v_norm = np.exp(log_v_tilde - logsumexp(log_v_tilde))           # (J,)

    # ── Step 6: ESS ──

    ESS = 1.0 / np.sum(v_norm ** 2)

    # ── Step 7: resample if ESS < 0.5 * J ──

    y_hat_minus_new = np.zeros((J, w))

    if ESS < 0.5 * J:
        idx = rng.choice(J, size=J, replace=True, p=v_norm)
        y_hat_minus_new[:, :w - 1] = y_hat_minus_prev[idx]
        y_hat_minus_new[:, w - 1]  = y_w_minus[idx]
        y_hat_plus_out = y_hat_plus_prev[idx].copy()
        v_hat_minus_new = np.full(J, 1.0 / J)
    else:
        y_hat_minus_new[:, :w - 1] = y_hat_minus_prev
        y_hat_minus_new[:, w - 1]  = y_w_minus
        y_hat_plus_out = y_hat_plus_prev.copy()
        v_hat_minus_new = v_norm.copy()

    return y_hat_minus_new, y_hat_plus_out, v_hat_minus_new


def estimate_belief_state_plus(
    w, J,
    y_hat_minus, y_hat_plus_prev, v_hat_minus,
    nu_Y, Gamma_Y, sigma2_Y,
    nu_tilde_Y, Gamma_tilde_Y, sigma2_tilde_Y,
    X_Y, X_tilde_Y,
    I_w, J_w,
    Y_w=None, tilde_Y_w=None,
    rng=None,
):
    """
    Estimate belief state b_w^+(y) on Sunday *night* of week w-1  (w >= 2).

    Three observation regimes (no resampling in this step):

      Case 1  (I_w=1, J_w=1) – Y_w fully observed:
        Draw theta^{Y,(j)}, set y_w^+ = Y_w,
        weight by phi(Y_w; X_Y^T theta^Y, sigma_Y^2), normalise.

      Case 2  (I_w=0, J_w=1) – only proxy tilde_Y_w observed:
        Draw theta^{tilde_Y,(j)}, set y_w^+ = y_w^-,
        weight by phi(tilde_Y_w; X_tY^T theta^tY, sigma_tY^2), normalise.

      Case 3  (otherwise) – neither observed:
        Set y_w^+ = y_w^-, keep weights unchanged.

    Parameters
    ----------
    w                : int  – current week (>= 2)
    J                : int  – number of particles
    y_hat_minus      : (J, w) array   – minus trajectories (from Algorithm 1)
    y_hat_plus_prev  : (J, w-1) array – plus trajectories through w-1
    v_hat_minus      : (J,) array     – normalised weights  hat{v}^-_w

    nu_Y, Gamma_Y, sigma2_Y               : Y model posterior and variance
    nu_tilde_Y, Gamma_tilde_Y, sigma2_tilde_Y : tilde_Y model posterior and variance

    X_Y       : (p_Y,) array  – design vector  X_{w-1}^Y
    X_tilde_Y : (p_tY,) array – design vector  X_{w-1}^{tilde_Y}

    I_w       : int            – 1 if Y_w is fully observed
    J_w       : int            – 1 if tilde_Y_w (or Y_w) is observed
    Y_w       : float or None  – observed primary outcome
    tilde_Y_w : float or None  – observed proxy outcome

    rng       : numpy.random.Generator or None

    Returns
    -------
    y_hat_plus_new : (J, w) array – plus trajectories extended to week w
    v_hat_plus_new : (J,) array   – normalised weights  hat{v}^+_w
    """
    if rng is None:
        rng = np.random.default_rng()

    y_hat_plus_new = np.zeros((J, w))
    y_hat_plus_new[:, :w - 1] = y_hat_plus_prev

    if I_w == 1 and J_w == 1:
        # ── Case 1: Y_w fully observed ──
        theta_Y = rng.multivariate_normal(nu_Y, Gamma_Y, size=J)  # (J, p_Y)
        y_hat_plus_new[:, w - 1] = Y_w

        log_w = np.log(np.maximum(v_hat_minus, 1e-300))
        mu_Y_pred = theta_Y @ X_Y                                 # (J,)
        log_Y_lik = norm.logpdf(
            Y_w, loc=mu_Y_pred, scale=np.sqrt(sigma2_Y)
        )
        log_v_tilde = log_w + log_Y_lik
        v_hat_plus_new = np.exp(log_v_tilde - logsumexp(log_v_tilde))

    elif I_w == 0 and J_w == 1:
        # ── Case 2: only proxy tilde_Y_w observed ──
        theta_tY = rng.multivariate_normal(
            nu_tilde_Y, Gamma_tilde_Y, size=J
        )                                                          # (J, p_tY)
        y_hat_plus_new[:, w - 1] = y_hat_minus[:, w - 1]

        log_w = np.log(np.maximum(v_hat_minus, 1e-300))
        mu_tY = theta_tY @ X_tilde_Y                              # (J,)
        log_tY_lik = norm.logpdf(
            tilde_Y_w, loc=mu_tY, scale=np.sqrt(sigma2_tilde_Y)
        )
        log_v_tilde = log_w + log_tY_lik
        v_hat_plus_new = np.exp(log_v_tilde - logsumexp(log_v_tilde))

    else:
        # ── Case 3: neither observed ──
        y_hat_plus_new[:, w - 1] = y_hat_minus[:, w - 1]
        v_hat_plus_new = v_hat_minus.copy()

    return y_hat_plus_new, v_hat_plus_new


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Helper functions for the online RL algorithm
# ──────────────────────────────────────────────────────────────────

def summarize_belief(y_hat, v_hat):
    """
    Extract point estimate and uncertainty of Y_w from particles.

    Parameters
    ----------
    y_hat : (J, w) array – particle trajectories
    v_hat : (J,)   array – normalised weights

    Returns
    -------
    b_hat   : float – weighted mean of current-week particles  (hat{b}_w)
    b_tilde : float – weighted std of current-week particles   (tilde{b}_w)
    """
    y_w = y_hat[:, -1]
    b_hat = np.average(y_w, weights=v_hat)
    b_tilde = np.sqrt(np.average((y_w - b_hat) ** 2, weights=v_hat))
    return b_hat, b_tilde


def ensemble_action_prob(phi_1, phi_0, betas):
    """
    Fraction of ensemble models that prefer action 1 over action 0.

        pi_hat = (1/B) sum_b  I( phi_1^T beta_b  >  phi_0^T beta_b )

    Parameters
    ----------
    phi_1  : (p,) array – features for action=1
    phi_0  : (p,) array – features for action=0
    betas  : list of B (p,) arrays – ensemble parameters

    Returns
    -------
    pi_hat : float in [0, 1]
    """
    B = len(betas)
    votes = sum(1 for beta_b in betas if phi_1 @ beta_b > phi_0 @ beta_b)
    return votes / B


def clip_prob(pi_hat, epsilon_0):
    """Clip randomisation probability to [epsilon_0, 1 - epsilon_0]."""
    return np.clip(pi_hat, epsilon_0, 1.0 - epsilon_0)


def compute_rlsvi_betas(Phi, targets_per_b, mu_0, Sigma_0, sigma2,
                        gamma_bar, z_prev, rng):
    """
    RLSVI with per-ensemble TD targets and discounted noise.

    Each ensemble member b has its own TD targets y^{(b)} (because
    targets depend on beta_prev^{(b)}), but shares the same feature
    matrix Phi.

    Closed-form posterior mean per ensemble member:

        Sigma_{w-1}     = (Sigma_0^{-1} + Phi^T Phi / sigma2)^{-1}
        mu_{w-1}^{(b)}  = Sigma_{w-1} (Sigma_0^{-1} mu_0
                                        + Phi^T y^{(b)} / sigma2)

    Discounted noise (AR(1) process across weeks):

        z_{w-1}^{(b)} ~ N(gamma_bar * z_{w-2}^{(b)},
                          (1 - gamma_bar^2) * Sigma_{w-1})

    Final parameter:  beta_{w-1}^{(b)} = mu_{w-1}^{(b)} + z_{w-1}^{(b)}

    Parameters
    ----------
    Phi           : (n, p) array – shared feature matrix  X_{w-1}
    targets_per_b : list of B (n,) arrays – per-ensemble TD targets y^{(b)}
    mu_0          : (p,) – prior mean
    Sigma_0       : (p, p) – prior covariance
    sigma2        : float – observation noise variance
    gamma_bar     : float – AR(1) coefficient for the noise discount
    z_prev        : list of B (p,) arrays – previous noise vectors z_{w-2}^{(b)}
    rng           : numpy.random.Generator

    Returns
    -------
    betas : list of B (p,) arrays – beta_{w-1}^{(b)}
    z_new : list of B (p,) arrays – updated noise vectors z_{w-1}^{(b)}
    """
    B = len(targets_per_b)
    Sigma_0_inv = np.linalg.inv(Sigma_0)

    if Phi.shape[0] == 0:
        Sigma_post = Sigma_0.copy()
    else:
        Sigma_post = np.linalg.inv(
            Sigma_0_inv + (1.0 / sigma2) * (Phi.T @ Phi))

    noise_cov = (1.0 - gamma_bar ** 2) * Sigma_post
    noise_cov = 0.5 * (noise_cov + noise_cov.T)
    min_eig = np.linalg.eigvalsh(noise_cov).min()
    if min_eig < 1e-6:
        noise_cov += (1e-6 - min_eig) * np.eye(noise_cov.shape[0])
    precomp = Sigma_0_inv @ mu_0

    betas, z_new = [], []
    for b in range(B):
        if Phi.shape[0] == 0:
            mu_b = mu_0.copy()
        else:
            mu_b = Sigma_post @ (
                precomp + (1.0 / sigma2) * (Phi.T @ targets_per_b[b]))
        try:
            z_b = rng.multivariate_normal(gamma_bar * z_prev[b], noise_cov)
        except np.linalg.LinAlgError:
            L = np.linalg.cholesky(noise_cov + 1e-4 * np.eye(noise_cov.shape[0]))
            z_b = gamma_bar * z_prev[b] + L @ rng.standard_normal(noise_cov.shape[0])
        betas.append(mu_b + z_b)
        z_new.append(z_b)

    return betas, z_new


# ──────────────────────────────────────────────────────────────────
# Shared helper: compute PF posteriors from initial priors
# ──────────────────────────────────────────────────────────────────

def _compute_pf_posteriors(w, pf_data, n_med,
                           nu_0_MY, Gamma_0_MY, sigma2_MY,
                           nu_0_Y, Gamma_0_Y, sigma2_Y,
                           nu_0_tilde_Y, Gamma_0_tilde_Y, sigma2_tilde_Y):
    """
    At w >= 3: posterior from initial priors + cumulative data through w-2.
    At w = 2: returns copies of the initial priors (no data yet).
    """
    if w >= 3:
        nu_MY_w, Gamma_MY_w = [], []
        for m in range(n_med):
            nu_m, G_m = bayesian_posterior_update(
                nu_0_MY[m], Gamma_0_MY[m],
                pf_data['X_cumul_MY'][m], pf_data['y_cumul_MY'][m],
                sigma2_MY[m],
            )
            nu_MY_w.append(nu_m)
            Gamma_MY_w.append(G_m)
        nu_Y_w, Gamma_Y_w = bayesian_posterior_update(
            nu_0_Y, Gamma_0_Y,
            pf_data['X_cumul_Y'], pf_data['y_cumul_Y'], sigma2_Y,
        )
        nu_tY_w, Gamma_tY_w = bayesian_posterior_update(
            nu_0_tilde_Y, Gamma_0_tilde_Y,
            pf_data['X_cumul_tY'], pf_data['y_cumul_tY'], sigma2_tilde_Y,
        )
    else:
        nu_MY_w = [nu.copy() for nu in nu_0_MY]
        Gamma_MY_w = [G.copy() for G in Gamma_0_MY]
        nu_Y_w = nu_0_Y.copy()
        Gamma_Y_w = Gamma_0_Y.copy()
        nu_tY_w = nu_0_tilde_Y.copy()
        Gamma_tY_w = Gamma_0_tilde_Y.copy()

    return nu_MY_w, Gamma_MY_w, nu_Y_w, Gamma_Y_w, nu_tY_w, Gamma_tY_w


# ──────────────────────────────────────────────────────────────────
# Feature map for walking-suggestion RL
# ──────────────────────────────────────────────────────────────────

def build_phi_action(b_hat, b_tilde, state, d, t, action):
    """
    Feature map  phi(tilde_S_{w,d,t}, A_{w,d,t}).

    phi = [1, d, t, E_w, d*E_w, t*E_w, b_w, d*b_w, t*b_w]           (9)
        ⌢ [tilde_M^Y_{1:6,1:2}, tilde_M^E_{1:6,1:2}, C_{w,d,t}]     (24 + n_c)
        ⌢ I(d=i,t=j) * A * [1, E_w, b_w, C_{w,d,t}]  ∀(i,j)        (12*(3+n_c))

    where tilde_M_{w,i,j} = I{(i,j) < (d,t)} * M_{w,i,j}  (mediators
    from slots strictly before the current one; later slots are zeroed).

    Parameters
    ----------
    b_hat   : float – belief point estimate  hat{b}_w
    b_tilde : float – belief uncertainty     tilde{b}_w  (unused here)
    state : dict
        'E_w'  : float          – engagement score
        'M_Y'  : (6, 2) array   – mediator-Y values  M^Y_{w,1:6,1:2}
        'M_E'  : (6, 2) array   – mediator-E values  M^E_{w,1:6,1:2}
        'C'    : (n_c,) array   – state for this decision point
                  (excludes day-of-week and morning/afternoon indicators)
    d       : int – day   (1-based, 1..6)
    t       : int – slot  (1-based, 1 or 2)
    action  : int – A_{w,d,t} in {0, 1}

    Returns
    -------
    phi : (p,) array   where  p = 9 + 24 + n_c + 12*(3 + n_c)
    """
    E_w = state['E_w']
    b_w = b_hat

    # ── mediator masking: keep slots (i,j) strictly before (d,t) ──
    M_Y = np.asarray(state['M_Y']).reshape(6, 2)
    M_E = np.asarray(state['M_E']).reshape(6, 2)

    mask = np.zeros((6, 2), dtype=bool)
    for i in range(6):
        for j in range(2):
            if (i + 1, j + 1) < (d, t):
                mask[i, j] = True

    M_Y_tilde = np.where(mask, M_Y, 0.0).ravel()       # (12,)
    M_E_tilde = np.where(mask, M_E, 0.0).ravel()       # (12,)

    # ── state for current decision point ──
    C_dt = np.asarray(state['C']).ravel()              # (n_c,)

    # ── part 1: base features ──
    base = np.array([
        1.0, d, t, E_w,
        d * E_w, t * E_w,
        b_w, d * b_w, t * b_w,
    ])

    # ── part 2: masked mediators + state ──
    med_ctx = np.concatenate([M_Y_tilde, M_E_tilde, C_dt])

    # ── part 3: action-interacted blocks (one per slot) ──
    interact_vec = np.concatenate([[1.0, E_w, b_w], C_dt])
    n_int = len(interact_vec)
    action_blocks = np.zeros(12 * n_int)
    if action == 1:
        slot_idx = (d - 1) * 2 + (t - 1)
        action_blocks[slot_idx * n_int : (slot_idx + 1) * n_int] = interact_vec

    return np.concatenate([base, med_ctx, action_blocks])


# ──────────────────────────────────────────────────────────────────
# RLSVI training data: feature matrix Phi and TD targets
# ──────────────────────────────────────────────────────────────────

def _next_slot(d, t):
    """Successor of (d, t) in lexicographic order, or None for (6, 2)."""
    if t == 1:
        return (d, 2)
    if d < 6:
        return (d + 1, 1)
    return None


def build_rl_training_data(w, A_hist, b_hat_hist, b_tilde_hist,
                           betas_eval, betas_select, gamma_dt,
                           get_state, reward_fn, phi_fn=None,
                           include_query=False, I_hist=None,
                           b_hat_query_hist=None, b_tilde_query_hist=None,
                           gamma_query=None):
    """
    Build the RLSVI feature matrix and per-ensemble TD targets.

    Stacks phi(S_{w',d,t}, A_{w',d,t}) for w'=1..w-1 and all 12 walking
    slots (d,t) in {(1,1),(1,2),...,(6,2)}.

    When include_query=True (Algorithm 2), each week also prepends a
    query row whose feature uses the **minus** belief and whose TD
    target bootstraps from the first walking action (d=1,t=1) using
    the **plus** belief passed via b_hat_hist / b_tilde_hist:

      y_query^{(b)} = gamma_query * phi(S_{w',1,1}, a*)^T beta_eval^{(b)}

    TD targets (per ensemble member b, double-Q style):

      a* = argmax_a  phi(S_next, a)^T  beta_select^{(b)}   (target net)

      Non-terminal (d,t) < (6,2):
        y^{(b)} = gamma_{d,t} * phi(S_next, a*)^T  beta_eval^{(b)}

      Terminal (d,t) = (6,2):
        if no query:
        y^{(b)} = R_{w'+1} + gamma_{6,2} * phi(S_next, a*)^T beta_eval^{(b)}
        if query:
        y^{(b)} = R_{w'+1} + gamma_{6,2} * phi(S_next, i*)^T beta_eval^{(b)}

    Parameters
    ----------
    w            : int – current week (training data from weeks 1 .. w-1)
    A_hist       : (W+1, 6, 2) int array – walking action history
    b_hat_hist   : (W+1,) array – belief means per week (plus belief for
                   Algorithm 2, combined belief for Algorithm 1)
    b_tilde_hist : (W+1,) array – belief stds per week
    betas_eval   : list of B (p,) arrays – betas for **value evaluation**
    betas_select : list of B (p,) arrays – betas for **action selection**
                   (target network, updated every C steps)
    gamma_dt     : (6, 2) array – per-slot discount factors
    get_state    : callable(w, d, t) -> dict
    reward_fn    : callable(w') -> float – surrogate reward R_{w'+1}
    phi_fn       : callable – feature map function
                   (default: build_phi_action)

    Query-specific (only when include_query=True)
    ----------------------------------------------
    include_query      : bool – if True, include query action in training data
    I_hist             : (W+1,) int array – query action history
    b_hat_query_hist   : (W+1,) array – minus belief means (for query features)
    b_tilde_query_hist : (W+1,) array – minus belief stds
    gamma_query        : float – discount factor for query → first walking

    Returns
    -------
    Phi           : (n_rows, p) array – feature matrix
                    n_rows = (12 + include_query) * (w - 1)
    targets_per_b : list of B (n_rows,) arrays – per-ensemble TD targets
    """
    if phi_fn is None:
        phi_fn = build_phi_action

    B = len(betas_eval)
    Phi_rows = []
    targets = [[] for _ in range(B)]

    for wp in range(1, w):
        bh_wp = b_hat_hist[wp]
        bt_wp = b_tilde_hist[wp]

        # ── optional query row (Algorithm 2) ──
        if include_query:
            bh_q = b_hat_query_hist[wp]
            bt_q = b_tilde_query_hist[wp]
            state_q = get_state(wp, 0, 0)
            a_q = I_hist[wp]
            Phi_rows.append(
                phi_fn(bh_q, bt_q, state_q, 0, 0, a_q, is_query=True))

            # bootstrap from first walking slot using plus belief
            state_11 = get_state(wp, 1, 1)
            phi_1 = phi_fn(bh_wp, bt_wp, state_11, 1, 1, 1)
            phi_0 = phi_fn(bh_wp, bt_wp, state_11, 1, 1, 0)
            for b in range(B):
                a_star = 1 if (phi_1 @ betas_select[b]
                               > phi_0 @ betas_select[b]) else 0
                q_star = (phi_1 if a_star else phi_0) @ betas_eval[b]
                targets[b].append(gamma_query * q_star)

        # ── 12 walking rows ──
        for d in range(1, 7):
            for t in range(1, 3):
                state_dt = get_state(wp, d, t)
                a_dt = A_hist[wp, d - 1, t - 1]
                Phi_rows.append(
                    phi_fn(bh_wp, bt_wp, state_dt, d, t, a_dt))

                nxt = _next_slot(d, t)

                if nxt is not None:
                    # non-terminal: bootstrap from same-week successor
                    d_n, t_n = nxt
                    state_n = get_state(wp, d_n, t_n)
                    phi_1 = phi_fn(bh_wp, bt_wp, state_n, d_n, t_n, 1)
                    phi_0 = phi_fn(bh_wp, bt_wp, state_n, d_n, t_n, 0)
                    for b in range(B):
                        a_star = 1 if (phi_1 @ betas_select[b]
                                       > phi_0 @ betas_select[b]) else 0
                        q_star = (phi_1 if a_star else phi_0) @ betas_eval[b]
                        targets[b].append(
                            gamma_dt[d - 1, t - 1] * q_star)
                else:
                    # terminal (6,2): reward + bootstrap from next week
                    R_next = reward_fn(wp)
                    bh_n = b_hat_hist[wp + 1]
                    bt_n = b_tilde_hist[wp + 1]
                    if include_query:
                        state_n = get_state(wp + 1, 0, 0)
                        phi_1 = phi_fn(bh_n, bt_n, state_n,
                                       0, 0, 1, is_query=True)
                        phi_0 = phi_fn(bh_n, bt_n, state_n,
                                       0, 0, 0, is_query=True)
                    else:
                        state_n = get_state(wp + 1, 1, 1)
                        phi_1 = phi_fn(bh_n, bt_n, state_n, 1, 1, 1)
                        phi_0 = phi_fn(bh_n, bt_n, state_n, 1, 1, 0)
                    for b in range(B):
                        a_star = 1 if (phi_1 @ betas_select[b]
                                       > phi_0 @ betas_select[b]) else 0
                        q_star = (phi_1 if a_star else phi_0) @ betas_eval[b]
                        targets[b].append(
                            R_next + gamma_dt[5, 1] * q_star)

    if Phi_rows:
        Phi = np.array(Phi_rows)
    else:
        p = len(betas_eval[0])
        Phi = np.empty((0, p))

    targets_per_b = [np.array(t) for t in targets]
    return Phi, targets_per_b


# # ──────────────────────────────────────────────────────────────────
# # Algorithm 1: Micro-randomize query, combined belief state
# # ──────────────────────────────────────────────────────────────────

# def online_rl_micro_query(
#     W, J, B, epsilon_0,
#     mu_0_rl, Sigma_0_rl, sigma2_rl,
#     gamma_dt, gamma_bar, target_update_C,
#     nu_0_MY, Gamma_0_MY, sigma2_MY,
#     nu_0_Y, Gamma_0_Y, sigma2_Y,
#     nu_0_tilde_Y, Gamma_0_tilde_Y, sigma2_tilde_Y,
#     Y_1,
#     get_pf_data, observe_outcome, get_state, reward_fn,
#     step_action=None,
#     rng=None,
# ):
#     """
#     Online RL for ADAPT RCT with micro-randomised query action.

#     Query is always Bernoulli(0.5) (I_w=1 at w=2).
#     Belief state is updated once per week using the combined PF.
#     Walking actions are split:
#       - Day 1 (d=1, t=1,2): uses beta_{w-2}
#       - Days 2-6: uses beta_{w-1} (freshly computed after day 1)

#     RLSVI uses per-ensemble TD targets and discounted noise (AR(1)):
#         beta_{w-1}^{(b)} = mu_{w-1}^{(b)} + z_{w-1}^{(b)}

#     TD targets use a double-Q style split:
#       - action selection (argmax): beta_{w-}  (target network,
#         copied every target_update_C steps)
#       - value evaluation:          beta_{w-1} (most recent betas)

#     Parameters
#     ----------
#     W, J, B       : int   – weeks, particles, ensemble size
#     epsilon_0     : float – clipping bound for randomisation probability

#     mu_0_rl       : (p_rl,)        – RL prior mean
#     Sigma_0_rl    : (p_rl, p_rl)   – RL prior covariance
#     sigma2_rl     : float           – RL noise variance
#     gamma_dt      : (6, 2) array    – per-slot discount factors
#     gamma_bar     : float           – AR(1) coefficient for noise discount
#     target_update_C : int           – update target network every C steps
#                       (C=1 means no lag, target = eval betas)

#     nu_0_MY, Gamma_0_MY, sigma2_MY : lists (length n_med) –
#         initial priors / noise for mediator models
#     nu_0_Y, Gamma_0_Y, sigma2_Y    : Y-model prior / noise
#     nu_0_tilde_Y, Gamma_0_tilde_Y, sigma2_tilde_Y : tilde-Y prior / noise

#     Y_1 : float – baseline affective association

#     Callbacks
#     ---------
#     get_pf_data(w) -> dict
#         PF inputs for week w.  Required keys:
#           'X_MY', 'M_Y_obs', 'X_Y', 'X_tilde_Y',
#           'X_cumul_MY', 'y_cumul_MY', 'X_cumul_Y', 'y_cumul_Y',
#           'X_cumul_tY', 'y_cumul_tY'

#     observe_outcome(w, I_w) -> (J_w, Y_w, tilde_Y_w)

#     get_state(w, d, t) -> dict
#         Return RL state for decision point (w, d, t).
#         Required keys: 'E_w', 'M_Y', 'M_E', 'C'.

#     reward_fn(w') -> float
#         Surrogate reward R_{w'+1} computed from week w' data.

#     Returns
#     -------
#     dict with keys: I, A, b_hat, b_tilde, pi_A, y_hat, v_hat, betas
#     """
#     if rng is None:
#         rng = np.random.default_rng()

#     n_med = len(nu_0_MY)
#     p_rl = mu_0_rl.shape[0]

#     # ── storage ──
#     I_hist = np.zeros(W + 1, dtype=int)
#     A_hist = np.zeros((W + 1, 6, 2), dtype=int)
#     b_hat_hist = np.full(W + 1, np.nan)
#     b_tilde_hist = np.full(W + 1, np.nan)
#     pi_A_hist = np.full((W + 1, 6, 2), np.nan)
#     betas_store = {}
#     z_store = {}

#     # ── week 1: initialise ──
#     A_hist[1] = rng.integers(0, 2, size=(6, 2))
#     I_hist[1] = 1
#     b_hat_hist[1] = Y_1
#     b_tilde_hist[1] = 0.0

#     if step_action is not None:
#         for _d in range(1, 7):
#             for _t in range(1, 3):
#                 step_action(1, _d, _t, int(A_hist[1, _d - 1, _t - 1]),
#                             I_hist[1])

#     y_hat = np.full((J, 1), Y_1)
#     v_hat = np.full(J, 1.0 / J)

#     # beta_0 drawn from prior (z_0 = 0)
#     z_init = [np.zeros(p_rl) for _ in range(B)]
#     empty_targets = [np.empty(0) for _ in range(B)]
#     betas_store[0], z_store[0] = compute_rlsvi_betas(
#         np.empty((0, p_rl)), empty_targets,
#         mu_0_rl, Sigma_0_rl, sigma2_rl, gamma_bar, z_init, rng,
#     )

#     # target network: beta_{w-}, updated every C steps
#     betas_target = betas_store[0]
#     steps_since_target_update = 0

#     # ── main loop: weeks 2 .. W ──
#     for w in range(2, W + 1):

#         # ──────── 1. query action: micro-randomised ────────
#         if w == 2:
#             I_w = 1
#         else:
#             I_w = rng.binomial(1, 0.5)
#         I_hist[w] = I_w

#         # ──────── 2. observe outcome ───────────────────────
#         J_w, Y_w, tilde_Y_w = observe_outcome(w, I_w)

#         # ──────── 3. PF posteriors + combined belief update ─
#         pf_data = get_pf_data(w)

#         (nu_MY_w, Gamma_MY_w,
#          nu_Y_w, Gamma_Y_w,
#          nu_tY_w, Gamma_tY_w) = _compute_pf_posteriors(
#             w, pf_data, n_med,
#             nu_0_MY, Gamma_0_MY, sigma2_MY,
#             nu_0_Y, Gamma_0_Y, sigma2_Y,
#             nu_0_tilde_Y, Gamma_0_tilde_Y, sigma2_tilde_Y,
#         )

#         y_hat, v_hat = estimate_belief_state(
#             w, J, y_hat, v_hat,
#             nu_MY_w, Gamma_MY_w, sigma2_MY,
#             nu_Y_w, Gamma_Y_w, sigma2_Y,
#             nu_tY_w, Gamma_tY_w, sigma2_tilde_Y,
#             pf_data['X_MY'], pf_data['M_Y_obs'],
#             pf_data['X_Y'], pf_data['X_tilde_Y'],
#             I_w, J_w,
#             Y_w=Y_w, tilde_Y_w=tilde_Y_w,
#             rng=rng,
#         )

#         b_hat_w, b_tilde_w = summarize_belief(y_hat, v_hat)
#         b_hat_hist[w] = b_hat_w
#         b_tilde_hist[w] = b_tilde_w

#         # ──────── 4. day-1 walking actions using beta_{w-2} ─
#         betas_old = betas_store.get(w - 2, betas_store[0])

#         for t in range(1, 3):
#             state_dt = get_state(w, 1, t)
#             phi_1 = build_phi_action(b_hat_w, b_tilde_w, state_dt, 1, t, 1)
#             phi_0 = build_phi_action(b_hat_w, b_tilde_w, state_dt, 1, t, 0)
#             pi_hat = ensemble_action_prob(phi_1, phi_0, betas_old)
#             pi_A = clip_prob(pi_hat, epsilon_0)
#             A_hist[w, 0, t - 1] = rng.binomial(1, pi_A)
#             pi_A_hist[w, 0, t - 1] = pi_A
#             if step_action is not None:
#                 step_action(w, 1, t, int(A_hist[w, 0, t - 1]), I_hist[w])

#         # ──────── 5. compute beta_{w-1} ─────────────────────
#         betas_eval = betas_store.get(w - 2, betas_store[0])
#         Phi_rl, targets_rl = build_rl_training_data(
#             w, A_hist, b_hat_hist, b_tilde_hist,
#             betas_eval, betas_target, gamma_dt,
#             get_state, reward_fn,
#         )
#         z_prev = z_store.get(w - 2, z_store[0])
#         betas_store[w - 1], z_store[w - 1] = compute_rlsvi_betas(
#             Phi_rl, targets_rl,
#             mu_0_rl, Sigma_0_rl, sigma2_rl, gamma_bar, z_prev, rng,
#         )

#         # update target network every C steps
#         steps_since_target_update += 1
#         if steps_since_target_update >= target_update_C:
#             betas_target = betas_store[w - 1]
#             steps_since_target_update = 0

#         # ──────── 6. days 2–6 walking actions using beta_{w-1}
#         betas_new = betas_store[w - 1]

#         for d in range(2, 7):
#             for t in range(1, 3):
#                 state_dt = get_state(w, d, t)
#                 phi_1 = build_phi_action(
#                     b_hat_w, b_tilde_w, state_dt, d, t, 1)
#                 phi_0 = build_phi_action(
#                     b_hat_w, b_tilde_w, state_dt, d, t, 0)
#                 pi_hat = ensemble_action_prob(phi_1, phi_0, betas_new)
#                 pi_A = clip_prob(pi_hat, epsilon_0)
#                 A_hist[w, d - 1, t - 1] = rng.binomial(1, pi_A)
#                 pi_A_hist[w, d - 1, t - 1] = pi_A
#                 if step_action is not None:
#                     step_action(w, d, t, int(A_hist[w, d - 1, t - 1]),
#                                 I_hist[w])

#     return {
#         'I': I_hist, 'A': A_hist,
#         'b_hat': b_hat_hist, 'b_tilde': b_tilde_hist,
#         'pi_A': pi_A_hist,
#         'y_hat': y_hat, 'v_hat': v_hat,
#         'betas': betas_store,
#     }


In [ ]:
class MicroQueryAgent:
    def __init__(
        self,
        W, J, B, epsilon_0,
        mu_0_rl, Sigma_0_rl, sigma2_rl,
        gamma_dt, gamma_bar, target_update_C,
        nu_0_MY, Gamma_0_MY, sigma2_MY,
        nu_0_Y, Gamma_0_Y, sigma2_Y,
        nu_0_tilde_Y, Gamma_0_tilde_Y, sigma2_tilde_Y,
        Y_1,
        get_state, reward_fn,
        rng=None,
    ):
        self.W = W
        self.J = J
        self.B = B
        self.epsilon_0 = epsilon_0
        self.mu_0_rl = mu_0_rl
        self.Sigma_0_rl = Sigma_0_rl
        self.sigma2_rl = sigma2_rl
        self.gamma_dt = gamma_dt
        self.gamma_bar = gamma_bar
        self.target_update_C = target_update_C

        self.nu_0_MY = nu_0_MY
        self.Gamma_0_MY = Gamma_0_MY
        self.sigma2_MY = sigma2_MY
        self.nu_0_Y = nu_0_Y
        self.Gamma_0_Y = Gamma_0_Y
        self.sigma2_Y = sigma2_Y
        self.nu_0_tilde_Y = nu_0_tilde_Y
        self.Gamma_0_tilde_Y = Gamma_0_tilde_Y
        self.sigma2_tilde_Y = sigma2_tilde_Y

        self.Y_1 = Y_1
        self.get_state = get_state
        self.reward_fn = reward_fn
        self.rng = np.random.default_rng() if rng is None else rng

    def reset(self):
        p_rl = self.mu_0_rl.shape[0]

        self.I_hist = np.zeros(self.W + 1, dtype=int)
        self.A_hist = np.zeros((self.W + 1, 6, 2), dtype=int)
        self.b_hat_hist = np.full(self.W + 1, np.nan)
        self.b_tilde_hist = np.full(self.W + 1, np.nan)
        self.pi_A_hist = np.full((self.W + 1, 6, 2), np.nan)
        self.betas_store = {}
        self.z_store = {}

        self.A_hist[1] = self.rng.integers(0, 2, size=(6, 2))
        self.I_hist[1] = 1
        self.b_hat_hist[1] = self.Y_1
        self.b_tilde_hist[1] = 0.0

        self.y_hat = np.full((self.J, 1), self.Y_1)
        self.v_hat = np.full(self.J, 1.0 / self.J)

        z_init = [np.zeros(p_rl) for _ in range(self.B)]
        empty_targets = [np.empty(0) for _ in range(self.B)]
        self.betas_store[0], self.z_store[0] = compute_rlsvi_betas(
            np.empty((0, p_rl)), empty_targets,
            self.mu_0_rl, self.Sigma_0_rl, self.sigma2_rl,
            self.gamma_bar, z_init, self.rng,
        )

        self.betas_target = self.betas_store[0]
        self.steps_since_target_update = 0
        self._current_betas_day1 = self.betas_store[0]
        self._current_betas_rest = None

    def begin_week(self, w, packet):
        if w == 1:
            return 1

        I_w = 1 if w == 2 else self.rng.binomial(1, 0.5)
        self.I_hist[w] = I_w

        J_w, Y_w, tilde_Y_w = outcome_from_packet(packet, I_w)
        pf_data = packet.pf_data

        n_med = len(self.nu_0_MY)
        (nu_MY_w, Gamma_MY_w,
         nu_Y_w, Gamma_Y_w,
         nu_tY_w, Gamma_tY_w) = _compute_pf_posteriors(
            w, pf_data, n_med,
            self.nu_0_MY, self.Gamma_0_MY, self.sigma2_MY,
            self.nu_0_Y, self.Gamma_0_Y, self.sigma2_Y,
            self.nu_0_tilde_Y, self.Gamma_0_tilde_Y, self.sigma2_tilde_Y,
        )

        self.y_hat, self.v_hat = estimate_belief_state(
            w, self.J, self.y_hat, self.v_hat,
            nu_MY_w, Gamma_MY_w, self.sigma2_MY,
            nu_Y_w, Gamma_Y_w, self.sigma2_Y,
            nu_tY_w, Gamma_tY_w, self.sigma2_tilde_Y,
            pf_data["X_MY"], pf_data["M_Y_obs"],
            pf_data["X_Y"], pf_data["X_tilde_Y"],
            I_w, J_w,
            Y_w=Y_w, tilde_Y_w=tilde_Y_w,
            rng=self.rng,
        )

        b_hat_w, b_tilde_w = summarize_belief(self.y_hat, self.v_hat)
        self.b_hat_hist[w] = b_hat_w
        self.b_tilde_hist[w] = b_tilde_w

        self._current_betas_day1 = self.betas_store.get(w - 2, self.betas_store[0])
        self._current_betas_rest = None
        return I_w

    def _compute_rest_of_week_beta(self, w):
        if self._current_betas_rest is not None:
            return

        betas_eval = self.betas_store.get(w - 2, self.betas_store[0])
        Phi_rl, targets_rl = build_rl_training_data(
            w, self.A_hist, self.b_hat_hist, self.b_tilde_hist,
            betas_eval, self.betas_target, self.gamma_dt,
            self.get_state, self.reward_fn,
        )
        z_prev = self.z_store.get(w - 2, self.z_store[0])

        self.betas_store[w - 1], self.z_store[w - 1] = compute_rlsvi_betas(
            Phi_rl, targets_rl,
            self.mu_0_rl, self.Sigma_0_rl, self.sigma2_rl,
            self.gamma_bar, z_prev, self.rng,
        )

        self.steps_since_target_update += 1
        if self.steps_since_target_update >= self.target_update_C:
            self.betas_target = self.betas_store[w - 1]
            self.steps_since_target_update = 0

        self._current_betas_rest = self.betas_store[w - 1]

    def act(self, w, d, t, state):
        if w == 1:
            self.pi_A_hist[1, d - 1, t - 1] = 0.5
            return int(self.A_hist[1, d - 1, t - 1])

        if d >= 2:
            self._compute_rest_of_week_beta(w)
            betas = self._current_betas_rest
        else:
            betas = self._current_betas_day1

        phi_1 = build_phi_action(self.b_hat_hist[w], self.b_tilde_hist[w], state, d, t, 1)
        phi_0 = build_phi_action(self.b_hat_hist[w], self.b_tilde_hist[w], state, d, t, 0)
        pi_hat = ensemble_action_prob(phi_1, phi_0, betas)
        pi_A = clip_prob(pi_hat, self.epsilon_0)
        A_wdt = self.rng.binomial(1, pi_A)

        self.A_hist[w, d - 1, t - 1] = A_wdt
        self.pi_A_hist[w, d - 1, t - 1] = pi_A
        return int(A_wdt)

    def results(self):
        return {
            "I": self.I_hist,
            "A": self.A_hist,
            "b_hat": self.b_hat_hist,
            "b_tilde": self.b_tilde_hist,
            "pi_A": self.pi_A_hist,
            "y_hat": self.y_hat,
            "v_hat": self.v_hat,
            "betas": self.betas_store,
        }

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Feature map for joint query + walking-action RL
# ──────────────────────────────────────────────────────────────────

def build_phi_action_query(b_hat, b_tilde, state, d, t, action,
                           is_query=False):
    """
    Shared feature map for joint query + walking-action RL.

    Same structure as build_phi_action, but with **13** action-interacted
    blocks instead of 12:

      block  0     : query block
      blocks 1-12  : walking blocks for (1,1),(1,2), … ,(6,2)

    When is_query = True  (query-action decision):
      • d and t are unused (set to 0 internally), so all d/t
        interactions in the base vanish.
      • M_Y and M_E are all zeroed (no mediators observed yet).
      • action fills the query block (block 0).
      • Context C comes from state['C_query'] (n_c,), or zeros if
        that key is absent.

    When is_query = False  (walking-action decision):
      • Identical to build_phi_action except the query block (block 0)
        is always zero and walking blocks are shifted by 1.

    phi = [1, d, t, E, d·E, t·E, b, d·b, t·b]                (9)
        ⌢ [tilde_M^Y (12), tilde_M^E (12), C_{w,d,t} (n_c)]  (24+n_c)
        ⌢ 13 × [1, E, b, C] action-interacted blocks          (13·(3+n_c))

    Parameters
    ----------
    b_hat   : float – belief point estimate  hat{b}_w
    b_tilde : float – belief uncertainty     tilde{b}_w  (unused here)
    state : dict
        'E_w'  : float          – engagement score
        'M_Y'  : (6, 2) array   – mediator-Y values  (ignored when is_query)
        'M_E'  : (6, 2) array   – mediator-E values  (ignored when is_query)
        'C'    : (n_c,) array   – state for this decision point
    d       : int – day   (1-based, 1..6; ignored when is_query=True)
    t       : int – slot  (1-based, 1 or 2; ignored when is_query=True)
    action  : int – 0 or 1
    is_query: bool – True for query decision, False for walking decision

    Returns
    -------
    phi : (p,) array   where  p = 9 + 24 + n_c + 13·(3 + n_c)
    """
    E_w = state['E_w']
    b_w = b_hat

    C_dt = np.asarray(state['C']).ravel()              # (n_c,)

    if is_query:
        d_feat, t_feat = 0.0, 0.0
        M_Y_tilde = np.zeros(12)
        M_E_tilde = np.zeros(12)
    else:
        d_feat, t_feat = float(d), float(t)

        M_Y = np.asarray(state['M_Y']).reshape(6, 2)
        M_E = np.asarray(state['M_E']).reshape(6, 2)
        mask = np.zeros((6, 2), dtype=bool)
        for i in range(6):
            for j in range(2):
                if (i + 1, j + 1) < (d, t):
                    mask[i, j] = True
        M_Y_tilde = np.where(mask, M_Y, 0.0).ravel()
        M_E_tilde = np.where(mask, M_E, 0.0).ravel()

    # ── part 1: base features ──
    base = np.array([
        1.0, d_feat, t_feat, E_w,
        d_feat * E_w, t_feat * E_w,
        b_w, d_feat * b_w, t_feat * b_w,
    ])

    # ── part 2: masked mediators + state ──
    med_ctx = np.concatenate([M_Y_tilde, M_E_tilde, C_dt])

    # ── part 3: action-interacted blocks (13 = 1 query + 12 walking) ──
    interact_vec = np.concatenate([[1.0, E_w, b_w], C_dt])
    n_int = len(interact_vec)
    action_blocks = np.zeros(13 * n_int)

    if action == 1:
        if is_query:
            action_blocks[:n_int] = interact_vec
        else:
            slot_idx = (d - 1) * 2 + (t - 1) + 1   # 1-12
            action_blocks[slot_idx * n_int
                          : (slot_idx + 1) * n_int] = interact_vec

    return np.concatenate([base, med_ctx, action_blocks])


# ──────────────────────────────────────────────────────────────────
# Algorithm 2: RL query action, twice-weekly belief state
# ──────────────────────────────────────────────────────────────────

# def online_rl_query(
#     W, J, B, epsilon_0,
#     mu_0_rl, Sigma_0_rl, sigma2_rl,
#     gamma_dt, gamma_bar, gamma_query, target_update_C,
#     nu_0_MY, Gamma_0_MY, sigma2_MY,
#     nu_0_Y, Gamma_0_Y, sigma2_Y,
#     nu_0_tilde_Y, Gamma_0_tilde_Y, sigma2_tilde_Y,
#     Y_1,
#     get_pf_data, observe_outcome, get_state, reward_fn,
#     step_action=None,
#     rng=None,
# ):
#     """
#     Online RL for ADAPT RCT with RL-decided query action.

#     Belief state is updated twice per week:
#       1. b_w^-  (mediator update via estimate_belief_state_minus)
#       2. I_w decided by RL ensemble using beta_{w-2} and b_w^-
#       3. b_w^+  (outcome update via estimate_belief_state_plus)
#     beta_w is computed after the plus update each week.
#     Walking actions for ALL (d,t) use beta_{w-1}.

#     RLSVI uses per-ensemble TD targets and discounted noise (AR(1)):
#         beta_w^{(b)} = mu_w^{(b)} + z_w^{(b)}

#     TD targets use a double-Q style split:
#       - action selection (argmax): beta_{w-}  (target network,
#         copied every target_update_C steps)
#       - value evaluation:          beta_{w-1} (most recent betas)

#     Parameters
#     ----------
#     W, J, B       : int   – weeks, particles, ensemble size
#     epsilon_0     : float – clipping bound for randomisation probability

#     mu_0_rl       : (p_rl,)        – RL prior mean
#     Sigma_0_rl    : (p_rl, p_rl)   – RL prior covariance
#     sigma2_rl     : float           – RL noise variance
#     gamma_dt      : (6, 2) array    – per-slot discount factors
#     gamma_bar     : float           – AR(1) coefficient for noise discount
#     gamma_query   : float           – discount factor for query → first walking
#     target_update_C : int           – update target network every C steps
#                       (C=1 means no lag, target = eval betas)

#     nu_0_MY, Gamma_0_MY, sigma2_MY : lists (length n_med) –
#         initial priors / noise for mediator models
#     nu_0_Y, Gamma_0_Y, sigma2_Y    : Y-model prior / noise
#     nu_0_tilde_Y, Gamma_0_tilde_Y, sigma2_tilde_Y : tilde-Y prior / noise

#     Y_1 : float – baseline affective association

#     Callbacks
#     ---------
#     get_pf_data(w) -> dict
#         PF inputs for week w.  Required keys:
#           'X_MY', 'M_Y_obs', 'X_Y', 'X_tilde_Y',
#           'X_cumul_MY', 'y_cumul_MY', 'X_cumul_Y', 'y_cumul_Y',
#           'X_cumul_tY', 'y_cumul_tY'

#     observe_outcome(w, I_w) -> (J_w, Y_w, tilde_Y_w)

#     get_state(w, d, t) -> dict
#         Return RL state for decision point (w, d, t).
#         For query: call with d=0, t=0.
#         Required keys: 'E_w', 'M_Y' (ignored for query),
#         'M_E' (ignored for query), 'C'.

#     reward_fn(w') -> float
#         Surrogate reward R_{w'+1} computed from week w' data.

#     Returns
#     -------
#     dict with keys: I, A, b_hat_minus, b_tilde_minus, b_hat_plus,
#                     b_tilde_plus, pi_I, pi_A,
#                     y_hat_minus, y_hat_plus, v_hat_plus, betas
#     """
#     if rng is None:
#         rng = np.random.default_rng()

#     n_med = len(nu_0_MY)
#     p_rl = mu_0_rl.shape[0]

#     # ── storage ──
#     I_hist = np.zeros(W + 1, dtype=int)
#     A_hist = np.zeros((W + 1, 6, 2), dtype=int)
#     b_hat_minus_hist = np.full(W + 1, np.nan)
#     b_tilde_minus_hist = np.full(W + 1, np.nan)
#     b_hat_plus_hist = np.full(W + 1, np.nan)
#     b_tilde_plus_hist = np.full(W + 1, np.nan)
#     pi_I_hist = np.full(W + 1, np.nan)
#     pi_A_hist = np.full((W + 1, 6, 2), np.nan)
#     betas_store = {}
#     z_store = {}

#     # ── week 1: initialise ──
#     A_hist[1] = rng.integers(0, 2, size=(6, 2))
#     I_hist[1] = 1
#     b_hat_plus_hist[1] = Y_1
#     b_tilde_plus_hist[1] = 0.0
#     b_hat_minus_hist[1] = Y_1
#     b_tilde_minus_hist[1] = 0.0

#     if step_action is not None:
#         for _d in range(1, 7):
#             for _t in range(1, 3):
#                 step_action(1, _d, _t, int(A_hist[1, _d - 1, _t - 1]),
#                             I_hist[1])

#     y_hat_minus = np.full((J, 1), Y_1)
#     y_hat_plus = np.full((J, 1), Y_1)
#     v_hat_plus = np.full(J, 1.0 / J)

#     # beta_0 drawn from prior (z_0 = 0);  beta_1 = beta_0 for w=2 walking
#     z_init = [np.zeros(p_rl) for _ in range(B)]
#     empty_targets = [np.empty(0) for _ in range(B)]
#     betas_store[0], z_store[0] = compute_rlsvi_betas(
#         np.empty((0, p_rl)), empty_targets,
#         mu_0_rl, Sigma_0_rl, sigma2_rl, gamma_bar, z_init, rng,
#     )
#     betas_store[1] = betas_store[0]
#     z_store[1] = z_store[0]

#     # target network: beta_{w-}, updated every C steps
#     betas_target = betas_store[0]
#     steps_since_target_update = 0

#     # ── main loop: weeks 2 .. W ──
#     for w in range(2, W + 1):

#         pf_data = get_pf_data(w)

#         # ──────── posteriors from initial priors + cumulative data ──
#         (nu_MY_w, Gamma_MY_w,
#          nu_Y_w, Gamma_Y_w,
#          nu_tY_w, Gamma_tY_w) = _compute_pf_posteriors(
#             w, pf_data, n_med,
#             nu_0_MY, Gamma_0_MY, sigma2_MY,
#             nu_0_Y, Gamma_0_Y, sigma2_Y,
#             nu_0_tilde_Y, Gamma_0_tilde_Y, sigma2_tilde_Y,
#         )

#         # ──────── 1. minus belief update (mediators only) ──────────
#         y_hat_minus, y_hat_plus, v_hat_minus = estimate_belief_state_minus(
#             w, J,
#             y_hat_minus, y_hat_plus, v_hat_plus,
#             nu_MY_w, Gamma_MY_w, sigma2_MY,
#             nu_Y_w, Gamma_Y_w, sigma2_Y,
#             pf_data['X_MY'], pf_data['M_Y_obs'], pf_data['X_Y'],
#             rng=rng,
#         )

#         b_hat_minus, b_tilde_minus = summarize_belief(
#             y_hat_minus, v_hat_minus)
#         b_hat_minus_hist[w] = b_hat_minus
#         b_tilde_minus_hist[w] = b_tilde_minus

#         # ──────── 2. decide query I_w via RL ensemble ──────────────
#         if w == 2:
#             I_w = 1
#             pi_I_w = 1.0
#         else:
#             state_q = get_state(w, 0, 0)
#             phi_q1 = build_phi_action_query(
#                 b_hat_minus, b_tilde_minus, state_q, 0, 0, 1,
#                 is_query=True)
#             phi_q0 = build_phi_action_query(
#                 b_hat_minus, b_tilde_minus, state_q, 0, 0, 0,
#                 is_query=True)
#             pi_hat_I = ensemble_action_prob(
#                 phi_q1, phi_q0, betas_store[w - 2])
#             pi_I_w = clip_prob(pi_hat_I, epsilon_0)
#             I_w = rng.binomial(1, pi_I_w)

#         I_hist[w] = I_w
#         pi_I_hist[w] = pi_I_w

#         # ──────── 3. observe outcome ───────────────────────────────
#         J_w, Y_w, tilde_Y_w = observe_outcome(w, I_w)

#         # ──────── 4. plus belief update (outcome information) ──────
#         y_hat_plus, v_hat_plus = estimate_belief_state_plus(
#             w, J,
#             y_hat_minus, y_hat_plus, v_hat_minus,
#             nu_Y_w, Gamma_Y_w, sigma2_Y,
#             nu_tY_w, Gamma_tY_w, sigma2_tilde_Y,
#             pf_data['X_Y'], pf_data['X_tilde_Y'],
#             I_w, J_w,
#             Y_w=Y_w, tilde_Y_w=tilde_Y_w,
#             rng=rng,
#         )

#         b_hat_plus, b_tilde_plus = summarize_belief(
#             y_hat_plus, v_hat_plus)
#         b_hat_plus_hist[w] = b_hat_plus
#         b_tilde_plus_hist[w] = b_tilde_plus

#         # ──────── 5. compute beta_w ────────────────────────────────
#         betas_eval = betas_store.get(w - 1, betas_store[0])
#         Phi_rl, targets_rl = build_rl_training_data(
#             w, A_hist, b_hat_plus_hist, b_tilde_plus_hist,
#             betas_eval, betas_target, gamma_dt,
#             get_state, reward_fn,
#             phi_fn=build_phi_action_query,
#             include_query=True, I_hist=I_hist,
#             b_hat_query_hist=b_hat_minus_hist,
#             b_tilde_query_hist=b_tilde_minus_hist,
#             gamma_query=gamma_query,
#         )
#         z_prev = z_store.get(w - 1, z_store[0])
#         betas_store[w], z_store[w] = compute_rlsvi_betas(
#             Phi_rl, targets_rl,
#             mu_0_rl, Sigma_0_rl, sigma2_rl, gamma_bar, z_prev, rng,
#         )

#         # update target network every C steps
#         steps_since_target_update += 1
#         if steps_since_target_update >= target_update_C:
#             betas_target = betas_store[w]
#             steps_since_target_update = 0

#         # ──────── 6. walking actions using beta_{w-1} ─────────────
#         betas_walk = betas_store[w - 1]

#         for d in range(1, 7):
#             for t in range(1, 3):
#                 state_dt = get_state(w, d, t)
#                 phi_1 = build_phi_action_query(
#                     b_hat_plus, b_tilde_plus, state_dt, d, t, 1)
#                 phi_0 = build_phi_action_query(
#                     b_hat_plus, b_tilde_plus, state_dt, d, t, 0)
#                 pi_hat = ensemble_action_prob(
#                     phi_1, phi_0, betas_walk)
#                 pi_A = clip_prob(pi_hat, epsilon_0)
#                 A_hist[w, d - 1, t - 1] = rng.binomial(1, pi_A)
#                 pi_A_hist[w, d - 1, t - 1] = pi_A
#                 if step_action is not None:
#                     step_action(w, d, t, int(A_hist[w, d - 1, t - 1]),
#                                 I_hist[w])

#     return {
#         'I': I_hist, 'A': A_hist,
#         'b_hat_minus': b_hat_minus_hist,
#         'b_tilde_minus': b_tilde_minus_hist,
#         'b_hat_plus': b_hat_plus_hist,
#         'b_tilde_plus': b_tilde_plus_hist,
#         'pi_I': pi_I_hist, 'pi_A': pi_A_hist,
#         'y_hat_minus': y_hat_minus, 'y_hat_plus': y_hat_plus,
#         'v_hat_plus': v_hat_plus,
#         'betas': betas_store,
#     }

In [ ]:
class RLQueryAgent:
    def __init__(
        self,
        W, J, B, epsilon_0,
        mu_0_rl, Sigma_0_rl, sigma2_rl,
        gamma_dt, gamma_bar, gamma_query, target_update_C,
        nu_0_MY, Gamma_0_MY, sigma2_MY,
        nu_0_Y, Gamma_0_Y, sigma2_Y,
        nu_0_tilde_Y, Gamma_0_tilde_Y, sigma2_tilde_Y,
        Y_1,
        get_state, reward_fn,
        rng=None,
    ):
        self.W = W
        self.J = J
        self.B = B
        self.epsilon_0 = epsilon_0
        self.mu_0_rl = mu_0_rl
        self.Sigma_0_rl = Sigma_0_rl
        self.sigma2_rl = sigma2_rl
        self.gamma_dt = gamma_dt
        self.gamma_bar = gamma_bar
        self.gamma_query = gamma_query
        self.target_update_C = target_update_C

        self.nu_0_MY = nu_0_MY
        self.Gamma_0_MY = Gamma_0_MY
        self.sigma2_MY = sigma2_MY
        self.nu_0_Y = nu_0_Y
        self.Gamma_0_Y = Gamma_0_Y
        self.sigma2_Y = sigma2_Y
        self.nu_0_tilde_Y = nu_0_tilde_Y
        self.Gamma_0_tilde_Y = Gamma_0_tilde_Y
        self.sigma2_tilde_Y = sigma2_tilde_Y

        self.Y_1 = Y_1
        self.get_state = get_state
        self.reward_fn = reward_fn
        self.rng = np.random.default_rng() if rng is None else rng

    def reset(self):
        p_rl = self.mu_0_rl.shape[0]

        self.I_hist = np.zeros(self.W + 1, dtype=int)
        self.A_hist = np.zeros((self.W + 1, 6, 2), dtype=int)
        self.b_hat_minus_hist = np.full(self.W + 1, np.nan)
        self.b_tilde_minus_hist = np.full(self.W + 1, np.nan)
        self.b_hat_plus_hist = np.full(self.W + 1, np.nan)
        self.b_tilde_plus_hist = np.full(self.W + 1, np.nan)
        self.pi_I_hist = np.full(self.W + 1, np.nan)
        self.pi_A_hist = np.full((self.W + 1, 6, 2), np.nan)
        self.betas_store = {}
        self.z_store = {}

        self.A_hist[1] = self.rng.integers(0, 2, size=(6, 2))
        self.I_hist[1] = 1
        self.b_hat_minus_hist[1] = self.Y_1
        self.b_tilde_minus_hist[1] = 0.0
        self.b_hat_plus_hist[1] = self.Y_1
        self.b_tilde_plus_hist[1] = 0.0

        self.y_hat_minus = np.full((self.J, 1), self.Y_1)
        self.y_hat_plus = np.full((self.J, 1), self.Y_1)
        self.v_hat_plus = np.full(self.J, 1.0 / self.J)

        z_init = [np.zeros(p_rl) for _ in range(self.B)]
        empty_targets = [np.empty(0) for _ in range(self.B)]
        self.betas_store[0], self.z_store[0] = compute_rlsvi_betas(
            np.empty((0, p_rl)), empty_targets,
            self.mu_0_rl, self.Sigma_0_rl, self.sigma2_rl,
            self.gamma_bar, z_init, self.rng,
        )
        self.betas_store[1] = self.betas_store[0]
        self.z_store[1] = self.z_store[0]

        self.betas_target = self.betas_store[0]
        self.steps_since_target_update = 0
        self._current_betas_walk = self.betas_store[0]

    def begin_week(self, w, packet):
        if w == 1:
            self.pi_I_hist[1] = 1.0
            return 1

        pf_data = packet.pf_data
        n_med = len(self.nu_0_MY)

        (nu_MY_w, Gamma_MY_w,
         nu_Y_w, Gamma_Y_w,
         nu_tY_w, Gamma_tY_w) = _compute_pf_posteriors(
            w, pf_data, n_med,
            self.nu_0_MY, self.Gamma_0_MY, self.sigma2_MY,
            self.nu_0_Y, self.Gamma_0_Y, self.sigma2_Y,
            self.nu_0_tilde_Y, self.Gamma_0_tilde_Y, self.sigma2_tilde_Y,
        )

        self.y_hat_minus, self.y_hat_plus, v_hat_minus = estimate_belief_state_minus(
            w, self.J,
            self.y_hat_minus, self.y_hat_plus, self.v_hat_plus,
            nu_MY_w, Gamma_MY_w, self.sigma2_MY,
            nu_Y_w, Gamma_Y_w, self.sigma2_Y,
            pf_data["X_MY"], pf_data["M_Y_obs"], pf_data["X_Y"],
            rng=self.rng,
        )

        b_hat_minus, b_tilde_minus = summarize_belief(self.y_hat_minus, v_hat_minus)
        self.b_hat_minus_hist[w] = b_hat_minus
        self.b_tilde_minus_hist[w] = b_tilde_minus

        if w == 2:
            I_w = 1
            pi_I_w = 1.0
        else:
            state_q = self.get_state(w, 0, 0)
            phi_q1 = build_phi_action_query(
                b_hat_minus, b_tilde_minus, state_q, 0, 0, 1, is_query=True
            )
            phi_q0 = build_phi_action_query(
                b_hat_minus, b_tilde_minus, state_q, 0, 0, 0, is_query=True
            )
            pi_hat_I = ensemble_action_prob(phi_q1, phi_q0, self.betas_store[w - 2])
            pi_I_w = clip_prob(pi_hat_I, self.epsilon_0)
            I_w = self.rng.binomial(1, pi_I_w)

        self.I_hist[w] = I_w
        self.pi_I_hist[w] = pi_I_w

        J_w, Y_w, tilde_Y_w = outcome_from_packet(packet, I_w)

        self.y_hat_plus, self.v_hat_plus = estimate_belief_state_plus(
            w, self.J,
            self.y_hat_minus, self.y_hat_plus, v_hat_minus,
            nu_Y_w, Gamma_Y_w, self.sigma2_Y,
            nu_tY_w, Gamma_tY_w, self.sigma2_tilde_Y,
            pf_data["X_Y"], pf_data["X_tilde_Y"],
            I_w, J_w,
            Y_w=Y_w, tilde_Y_w=tilde_Y_w,
            rng=self.rng,
        )

        b_hat_plus, b_tilde_plus = summarize_belief(self.y_hat_plus, self.v_hat_plus)
        self.b_hat_plus_hist[w] = b_hat_plus
        self.b_tilde_plus_hist[w] = b_tilde_plus

        betas_eval = self.betas_store.get(w - 1, self.betas_store[0])
        Phi_rl, targets_rl = build_rl_training_data(
            w, self.A_hist,
            self.b_hat_plus_hist, self.b_tilde_plus_hist,
            betas_eval, self.betas_target, self.gamma_dt,
            self.get_state, self.reward_fn,
            phi_fn=build_phi_action_query,
            include_query=True,
            I_hist=self.I_hist,
            b_hat_query_hist=self.b_hat_minus_hist,
            b_tilde_query_hist=self.b_tilde_minus_hist,
            gamma_query=self.gamma_query,
        )
        z_prev = self.z_store.get(w - 1, self.z_store[0])
        self.betas_store[w], self.z_store[w] = compute_rlsvi_betas(
            Phi_rl, targets_rl,
            self.mu_0_rl, self.Sigma_0_rl, self.sigma2_rl,
            self.gamma_bar, z_prev, self.rng,
        )

        self.steps_since_target_update += 1
        if self.steps_since_target_update >= self.target_update_C:
            self.betas_target = self.betas_store[w]
            self.steps_since_target_update = 0

        self._current_betas_walk = self.betas_store[w - 1]
        return I_w

    def act(self, w, d, t, state):
        if w == 1:
            self.pi_A_hist[1, d - 1, t - 1] = 0.5
            return int(self.A_hist[1, d - 1, t - 1])

        phi_1 = build_phi_action_query(
            self.b_hat_plus_hist[w], self.b_tilde_plus_hist[w], state, d, t, 1
        )
        phi_0 = build_phi_action_query(
            self.b_hat_plus_hist[w], self.b_tilde_plus_hist[w], state, d, t, 0
        )
        pi_hat = ensemble_action_prob(phi_1, phi_0, self._current_betas_walk)
        pi_A = clip_prob(pi_hat, self.epsilon_0)
        A_wdt = self.rng.binomial(1, pi_A)

        self.A_hist[w, d - 1, t - 1] = A_wdt
        self.pi_A_hist[w, d - 1, t - 1] = pi_A
        return int(A_wdt)

    def results(self):
        return {
            "I": self.I_hist,
            "A": self.A_hist,
            "b_hat_minus": self.b_hat_minus_hist,
            "b_tilde_minus": self.b_tilde_minus_hist,
            "b_hat_plus": self.b_hat_plus_hist,
            "b_tilde_plus": self.b_tilde_plus_hist,
            "pi_I": self.pi_I_hist,
            "pi_A": self.pi_A_hist,
            "y_hat_minus": self.y_hat_minus,
            "y_hat_plus": self.y_hat_plus,
            "v_hat_plus": self.v_hat_plus,
            "betas": self.betas_store,
        }

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Callback factory: build get_pf_data, observe_outcome, get_state
# for a single participant in the simulation testbed
# ──────────────────────────────────────────────────────────────────

def make_callbacks(
    # ── raw data arrays (length N = n_weeks * K, one row per decision time) ──
    fourSC,                   # (N,) 4-hour step count  M^Y_{w,d,t}
    fourSC_cond,              # (N, p_fourSC) design matrix for fourSC model
    antic_affect,             # (N,) anticipated affect  M^Y_{w,d} (daily)
    antic_affect_cond,        # (N, p_antic) design matrix for antic. affect
    hourly_pageview,          # (N,) hourly pageview count  M^E_{w,d,t}
    hourly_pageview_cond,     # (N, p_hpv) design matrix for pageview model
    CAE,                      # (n_weeks,) CAE outcome  Y_w  (weekly level)
    CAE_cond,                 # (n_weeks, p_CAE) design matrix for Y model
    CAE_short,                # (n_weeks,) short-form CAE  tilde_Y_w
    CAE_short_cond,           # (n_weeks, p_tY) design matrix for tilde_Y
    perceived_utility,        # (n_weeks,) engagement  E_w
    # ── noise variances (from fitted models) ──
    sigma2_fourSC,
    sigma2_antic,
    sigma2_hpv,
    sigma2_CAE,
    sigma2_CAE_short,
    # ── layout ──
    K=14,                     # decision times per week (7 days × 2 slots)
    ar1_col=1,                # column index of AR(1) term in fourSC_cond
):
    """
    Create the three callback closures for one participant.

    Data layout convention
    ----------------------
    Row index in the (N,) arrays:
        row = w * K + (day - 1) * 2 + (t - 1)
    where w is 0-based week, day ∈ 1..7, t ∈ {1, 2}.
    Days 1–6 are action days; day 7 is observation-only (Sunday).

    Returns
    -------
    get_pf_data      : callable(w) -> dict
    observe_outcome   : callable(w, I_w) -> (J_w, Y_w, tilde_Y_w)
    get_state         : callable(w, d, t) -> dict
    """

    n_weeks = len(CAE)

    # ── helper: row index for (week, day, slot) ──
    def _row(w, d, t):
        """w is 1-based week, d ∈ 1..7, t ∈ {1, 2}."""
        return (w - 1) * K + (d - 1) * 2 + (t - 1)

    # ── helper: all row indices for days 1-6 of a given week ──
    def _week_action_rows(w):
        """Rows for d=1..6, t=1,2 of week w (12 rows)."""
        start = (w - 1) * K
        return list(range(start, start + 12))

    # ────────────────────────────────────────────────
    # 1. get_pf_data(w) — particle-filter inputs
    # ────────────────────────────────────────────────
    def get_pf_data(w):
        """
        Return PF inputs for the week-w belief update.

        The mediator observations and design vectors refer to week w-1
        (the most recently completed week).
        Cumulative data goes through week w-2.
        """
        # rows for week w-1 (the just-completed action week)
        prev_rows = _week_action_rows(w - 1)  # 12 rows for d=1..6

        # --- mediator design vectors and observations (week w-1) ---
        # n_med = 12 (6 days × 2 slots); same model form but
        # different covariate values per slot.
        # For slot 0 (d=1,t=1): zero out the AR(1) column.
        X_MY = []
        M_Y_obs = []
        for m, r in enumerate(prev_rows):
            x = fourSC_cond[r].copy()
            if m == 0:
                x[ar1_col] = 0.0
            X_MY.append(x)
            M_Y_obs.append(fourSC[r])             # NaN if missing

        # --- Y-model design vector (week w-1, weekly level) ---
        X_Y = CAE_cond[w - 2]        # w is 1-based; week w-1 → index w-2
        X_tilde_Y = CAE_short_cond[w - 2]

        # --- cumulative data through week w-2 ---
        # For w=2 the caller uses priors directly; these are only
        # accessed for w >= 3.
        if w >= 3:
            # Each mediator slot m trains on its OWN rows across
            # past weeks (same slot position, different weeks).
            X_cumul_MY = []
            y_cumul_MY = []
            for m in range(12):
                slot_rows = [(ww - 1) * K + m
                             for ww in range(1, w - 1)]
                X_m = fourSC_cond[slot_rows].copy()
                y_m = fourSC[slot_rows].copy()
                if m == 0:
                    X_m[:, ar1_col] = 0.0
                X_cumul_MY.append(X_m)
                y_cumul_MY.append(y_m)

            X_cumul_Y = CAE_cond[:w - 2]     # weeks 0 .. w-3 (0-based)
            y_cumul_Y = CAE[:w - 2]
            X_cumul_tY = CAE_short_cond[:w - 2]
            y_cumul_tY = CAE_short[:w - 2]
        else:
            X_cumul_MY = y_cumul_MY = None
            X_cumul_Y = y_cumul_Y = None
            X_cumul_tY = y_cumul_tY = None

        return {
            'X_MY': X_MY,
            'M_Y_obs': M_Y_obs,
            'X_Y': X_Y,
            'X_tilde_Y': X_tilde_Y,
            'X_cumul_MY': X_cumul_MY,
            'y_cumul_MY': y_cumul_MY,
            'X_cumul_Y': X_cumul_Y,
            'y_cumul_Y': y_cumul_Y,
            'X_cumul_tY': X_cumul_tY,
            'y_cumul_tY': y_cumul_tY,
        }

    # ────────────────────────────────────────────────
    # 2. observe_outcome(w, I_w) — outcome observation
    # ────────────────────────────────────────────────
    def observe_outcome(w, I_w):
        """
        Return (J_w, Y_w, tilde_Y_w) for week w.

        I_w = 1 → queried: attempt to observe the full CAE  (Y_w).
        I_w = 0 → not queried: attempt to observe short CAE (tilde_Y_w).
        J_w = 1 if the relevant outcome is actually available (non-NaN).
        """
        idx = w - 1                 # 0-based week index

        if I_w == 1:
            Y_w = CAE[idx] if idx < n_weeks else None
            J_w = 0 if (Y_w is None or np.isnan(Y_w)) else 1
            return (J_w, Y_w if J_w else None, None)
        else:
            tY_w = CAE_short[idx] if idx < n_weeks else None
            J_w = 0 if (tY_w is None or np.isnan(tY_w)) else 1
            return (J_w, None, tY_w if J_w else None)

    # ────────────────────────────────────────────────
    # 3. get_state(w, d, t) — RL state for a decision point
    # ────────────────────────────────────────────────
    def get_state(w, d, t):
        """
        Return state dict for decision point (w, d, t).

        For query decision (Algorithm 2): call with d=0, t=0.
        For walking decision: d ∈ 1..6, t ∈ {1, 2}.

        Keys
        ----
        'E_w'  : float     – perceived utility (engagement) for week w
        'M_Y'  : (6, 2)    – 4-hour step counts observed so far this week
                              (slots not yet reached are NaN)
        'M_E'  : (6, 2)    – hourly pageview counts observed so far
        'C'    : (n_c,)    – additional context for this slot
        """
        wk_idx = w - 1  # 0-based

        # E_w: engagement score for this week
        E_w = perceived_utility[wk_idx] if wk_idx < n_weeks else 0.0

        if d == 0 and t == 0:
            # query-level state: no mediators available yet
            M_Y = np.zeros((6, 2))
            M_E = np.zeros((6, 2))
            C = np.array([])   # TODO: add query-level context features
            return {'E_w': E_w, 'M_Y': M_Y, 'M_E': M_E, 'C': C}

        # fill mediator arrays with values from slots before (d, t)
        M_Y = np.full((6, 2), np.nan)
        M_E = np.full((6, 2), np.nan)
        for dd in range(1, 7):
            for tt in range(1, 3):
                if (dd, tt) < (d, t):
                    r = _row(w, dd, tt)
                    M_Y[dd - 1, tt - 1] = fourSC[r]
                    M_E[dd - 1, tt - 1] = hourly_pageview[r]

        # replace remaining NaN with 0 (unseen slots)
        M_Y = np.nan_to_num(M_Y, nan=0.0)
        M_E = np.nan_to_num(M_E, nan=0.0)

        # C: context features (exclude d, t — those are in phi already)
        # TODO: customise with your time-varying features
        C = np.array([])

        return {'E_w': E_w, 'M_Y': M_Y, 'M_E': M_E, 'C': C}

    return get_pf_data, observe_outcome, get_state